[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch13_grpo_ru.ipynb)

<div style="background: linear-gradient(135deg, #006494 0%, #20808D 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">Глава 13. GRPO: групповая относительная оптимизация стратегии</h1>
  <p style="color: #BCE2E7; margin-top: 10px; font-size: 1.1em;">
    GRPO заменяет критика-функцию-ценности групповой относительной оценкой преимущества,
    что хорошо подходит для задач с проверяемыми вознаграждениями (математика, код).
    Реализуем с нуля и сравниваем с REINFORCE и RLOO.
  </p>
</div>

**Цели обучения:**
- Реализовать оценку преимущества GRPO (групповая нормализация)
- Обучить на игрушечной математической задаче с бинарным верификатором
- Показать, как размер группы $K$ влияет на дисперсию преимуществ
- Сравнить GRPO, REINFORCE и RLOO на задаче бандита
- Построить распределения преимуществ для разных $K$

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 10px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Подготовка окружения</h2>
  <p style="color: #CDCCCA; margin-top: 8px;">Большинство экспериментов в этом ноутбуке CPU-friendly (бандиты). GPU ускоряет демо с языковой моделью на математике. Ожидаемое суммарное время: ~7 минут.</p>
</div>

In [ ]:
!pip install -q transformers accelerate datasets trl torch matplotlib numpy tqdm

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import trange
import warnings, re
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

plt.rcParams.update({
    'figure.facecolor': '#F7F6F2', 'axes.facecolor': '#F9F8F5',
    'axes.edgecolor': '#D4D1CA', 'axes.labelcolor': '#28251D',
    'text.color': '#28251D', 'xtick.color': '#7A7974',
    'ytick.color': '#7A7974', 'grid.color': '#D4D1CA',
    'font.family': 'sans-serif',
})

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§13.1 Алгоритм GRPO</h2>
</div>

### Мотивация

Стандартные алгоритмы градиента стратегии (PPO, REINFORCE) требуют **критика** (функцию ценности) для уменьшения дисперсии. Обучение критика вместе со стратегией добавляет сложность и вычисления.

**GRPO** (Shao и др., 2024 — используется в DeepSeek-R1) убирает критика так:

1. Для каждого входа $x$ сэмплируем **группу** из $K$ ответов $\{y_1, \ldots, y_K\}$.
2. Вычисляем вознаграждения $\{r_1, \ldots, r_K\}$ через верификатор.
3. **Нормализуем внутри группы**, получая преимущества:

$$ A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r}) + \epsilon} $$

4. Оптимизируем clipped-цель (как в PPO) с этими групповыми преимуществами.

Базовая линия неявная — она исходит из **остальных ответов в группе**, а не из обученной функции ценности.

In [ ]:
# ── GRPO Advantage Estimation ─────────────────────────────────────────────

def grpo_advantages(rewards: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Compute GRPO group-relative advantages.
    rewards: shape (K,) for one group, or (B, K) for a batch of groups.
    """
    if rewards.ndim == 1:
        rewards = rewards[np.newaxis, :]  # (1, K)
    mean = rewards.mean(axis=1, keepdims=True)   # (B, 1)
    std  = rewards.std(axis=1, keepdims=True)    # (B, 1)
    return (rewards - mean) / (std + eps)        # (B, K)


# Illustrate with a concrete example
K = 8
rewards_example = np.array([1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0])
advantages_example = grpo_advantages(rewards_example).squeeze()

print('Group rewards  :', rewards_example)
print('GRPO advantages:', advantages_example.round(3))
print(f'Mean adv: {advantages_example.mean():.6f} (should be ≈ 0)')
print(f'Std  adv: {advantages_example.std():.6f}  (should be ≈ 1)')

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§13.2 Игрушечная математическая задача с GRPO</h2>
</div>

Имитируем GRPO-цикл на простой арифметической задаче: модель должна сгенерировать двузначный ответ на вопросы о сложении однозначных чисел. **Верификатор** проверяет, совпадает ли число с правильным ответом.

Моделируем стратегию как небольшое категориальное распределение по строкам цифр и оптимизируем её GRPO.

In [ ]:
# ── Toy Math Task ─────────────────────────────────────────────────────────
# Policy: softmax over K candidate answer "strategies" (buckets)
# Reward: +1 if correct, 0 otherwise

np.random.seed(0)

class ToyMathPolicy(nn.Module):
    """Tiny policy: 1 hidden-layer network that outputs K logits (answer bins)."""
    def __init__(self, n_bins=20):
        super().__init__()
        self.n_bins = n_bins
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, n_bins)
        )

    def forward(self, x):
        return self.net(x)

    def sample_group(self, x, K):
        logits = self(x)
        probs = F.softmax(logits, dim=-1)
        samples = torch.multinomial(probs.expand(K, -1), 1).squeeze(-1)  # (K,)
        log_probs = F.log_softmax(logits, dim=-1)                         # (n_bins,)
        sampled_lp = log_probs[samples]                                   # (K,)
        return samples, sampled_lp


def math_verifier(samples, answer_bin, n_bins=20):
    """Return binary reward: 1 if sample == correct bin."""
    return (samples == answer_bin).float().numpy()


def run_grpo_math(K=8, n_steps=300, lr=5e-3, clip_eps=0.2):
    policy = ToyMathPolicy(n_bins=20)
    opt = Adam(policy.parameters(), lr=lr)

    rewards_per_step = []
    correct_rate = []

    for step in range(n_steps):
        # Random single-digit addition: a + b, answer in [0, 18]
        a, b = np.random.randint(0, 10), np.random.randint(0, 10)
        answer = a + b
        answer_bin = torch.tensor(answer)  # answer is 0-18, treat as bin index
        x = torch.tensor([a / 9.0, b / 9.0], dtype=torch.float32)

        samples, log_probs = policy.sample_group(x, K)

        rewards = torch.tensor(math_verifier(samples, answer_bin), dtype=torch.float32)
        advantages = torch.tensor(
            grpo_advantages(rewards.numpy()).squeeze(), dtype=torch.float32
        )

        # GRPO policy gradient loss (no clipping for simplicity)
        loss = -(advantages * log_probs).mean()
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
        opt.step()

        rewards_per_step.append(rewards.mean().item())
        correct_rate.append(rewards.max().item())  # 1 if any in group correct

    return rewards_per_step, correct_rate, policy


rewards_k8, correct_k8, policy_k8 = run_grpo_math(K=8)
print(f'K=8: Final avg reward (last 50 steps): {np.mean(rewards_k8[-50:]):.3f}')

In [ ]:
# Run for multiple K values
results = {}
for K in [1, 2, 4, 8, 16]:
    torch.manual_seed(42)
    np.random.seed(42)
    rews, _, _ = run_grpo_math(K=K, n_steps=300)
    results[K] = rews
    print(f'K={K:2d}: Final avg reward = {np.mean(rews[-50:]):.3f}')

In [ ]:
# Plot learning curves
smooth = lambda x, w=15: np.convolve(x, np.ones(w)/w, mode='valid')
colors = ['#BCE2E7', '#20808D', '#1B474D', '#A84B2F', '#944454']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for (K, rews), col in zip(results.items(), colors):
    smoothed = smooth(rews)
    axes[0].plot(smoothed, color=col, linewidth=2, label=f'K={K}')

axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Average Reward per Group')
axes[0].set_title('§13.2 — GRPO: Group Size K vs Learning Speed', fontweight='bold')
axes[0].legend(title='Group Size K')
axes[0].grid(alpha=0.4)

# Final performance vs K
Ks = list(results.keys())
final_rewards = [np.mean(results[K][-50:]) for K in Ks]
axes[1].plot(Ks, final_rewards, 'o-', color='#20808D', linewidth=2, markersize=8)
axes[1].set_xlabel('Group Size K')
axes[1].set_ylabel('Final Avg Reward (last 50 steps)')
axes[1].set_title('Effect of Group Size K on Final Performance', fontweight='bold')
axes[1].set_xticks(Ks)
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('ch13_fig1_grpo_k_effect.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§13.3 Распределение преимуществ при разных $K$</h2>
</div>

Дисперсия преимуществ убывает с ростом $K$, потому что группа даёт лучшую оценку базовой линии. При $K=1$ GRPO вырождается в нулевое преимущество (один сэмпл нельзя нормализовать). Это мотивирует использовать умеренно большие группы.

In [ ]:
# Simulate advantage distributions for different K values
# Assume reward is Bernoulli(p=0.3) for illustration
np.random.seed(7)
N_SAMPLES = 5000
P_CORRECT = 0.3

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)

K_values = [2, 4, 8, 16]
variances = []

for ax, K in zip(axes, K_values):
    all_advantages = []
    for _ in range(N_SAMPLES):
        group_rewards = np.random.binomial(1, P_CORRECT, K).astype(float)
        advs = grpo_advantages(group_rewards).squeeze()
        all_advantages.extend(advs.tolist())
    all_advantages = np.array(all_advantages)
    var = np.var(all_advantages[np.isfinite(all_advantages)])
    variances.append(var)

    finite_adv = all_advantages[np.isfinite(all_advantages)]
    ax.hist(finite_adv, bins=30, density=True, color='#20808D', alpha=0.8, edgecolor='white')
    ax.set_title(f'K = {K}\nVar = {var:.3f}', fontweight='bold')
    ax.set_xlabel('Advantage')
    if ax == axes[0]:
        ax.set_ylabel('Density')

plt.suptitle('§13.3 — Advantage Distributions for Different Group Sizes (p=0.3 binary reward)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('ch13_fig2_advantage_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

print('Advantage variance by K:')
for K, v in zip(K_values, variances):
    print(f'  K={K:2d}: {v:.4f}')

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§13.4 GRPO vs REINFORCE vs RLOO на бандите</h2>
</div>

Сравниваем три оценщика градиента стратегии на **гауссовском бандите** (непрерывное действие):

| Алгоритм | Базовая линия | Заметки |
|-----------|----------|-------|
| **REINFORCE** | Нет (сырое вознаграждение) | Самая большая дисперсия |
| **REINFORCE с базовой линией** | Скользящее среднее | Меньшая дисперсия |
| **RLOO** (Leave-one-out) | Среднее по остальным $K-1$ сэмплам | Несмещённый; как GRPO, но другая нормализация |
| **GRPO** | Нормализация по среднему и std группы | Стабилизированные преимущества |

**Преимущество RLOO** для сэмпла $i$:
$$ A_i^{\text{RLOO}} = r_i - \frac{1}{K-1}\sum_{j \neq i} r_j $$

In [ ]:
# ── K-armed Gaussian Bandit ───────────────────────────────────────────────
# True reward: r(a) = -0.5*(a - a*)^2, a* = 2.0
# Policy: Gaussian with learned mean μ, fixed std σ=1.0

TRUE_OPT = 2.0
N_STEPS_BANDIT = 800
K_BANDIT = 8
SIGMA = 1.0  # fixed policy std
LR_BANDIT = 0.02

def bandit_reward(a):
    return -0.5 * (a - TRUE_OPT) ** 2

def rloo_advantages(rewards):
    """Leave-one-out baseline."""
    K = len(rewards)
    advs = np.zeros(K)
    for i in range(K):
        others = np.concatenate([rewards[:i], rewards[i+1:]])
        advs[i] = rewards[i] - others.mean()
    return advs


def run_bandit(method='grpo', K=K_BANDIT, n_steps=N_STEPS_BANDIT, lr=LR_BANDIT, seed=42):
    np.random.seed(seed)
    mu = np.array([0.0])  # learnable parameter
    baseline_ema = 0.0
    reward_history = []
    mu_history = []

    for step in range(n_steps):
        # Sample K actions from Gaussian(mu, sigma)
        actions = mu[0] + SIGMA * np.random.randn(K)
        rewards = np.array([bandit_reward(a) for a in actions])

        if method == 'reinforce':
            advantages = rewards  # no baseline
        elif method == 'reinforce_baseline':
            baseline_ema = 0.95 * baseline_ema + 0.05 * rewards.mean()
            advantages = rewards - baseline_ema
        elif method == 'rloo':
            advantages = rloo_advantages(rewards)
        elif method == 'grpo':
            advantages = grpo_advantages(rewards).squeeze()

        # Policy gradient for Gaussian: ∂logπ/∂μ = (a - μ) / σ²
        grad = np.mean(advantages * (actions - mu[0]) / SIGMA**2)
        mu += lr * grad

        reward_history.append(rewards.mean())
        mu_history.append(mu[0])

    return reward_history, mu_history


methods = ['reinforce', 'reinforce_baseline', 'rloo', 'grpo']
labels  = ['REINFORCE', 'REINFORCE + baseline', 'RLOO', 'GRPO']
colors  = ['#A84B2F', '#FFC553', '#006494', '#20808D']
styles  = [':', '--', '-.', '-']

all_results = {}
for method in methods:
    rews, mus = run_bandit(method=method, seed=42)
    all_results[method] = {'rewards': rews, 'mus': mus}
    print(f'{method:25s}: final μ={mus[-1]:.4f} (opt={TRUE_OPT}), '
          f'final avg reward={np.mean(rews[-50:]):.4f}')

In [ ]:
smooth = lambda x, w=20: np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for method, label, col, ls in zip(methods, labels, colors, styles):
    r = all_results[method]
    axes[0].plot(smooth(r['rewards']), color=col, linewidth=2,
                 linestyle=ls, label=label)
    axes[1].plot(smooth(r['mus']), color=col, linewidth=2,
                 linestyle=ls, label=label)

axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Average Reward')
axes[0].set_title('§13.4 — Bandit: Reward per Step (K=8)', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.4)

axes[1].axhline(TRUE_OPT, color='#28251D', linestyle=':', linewidth=1.5,
                label=f'Optimal μ={TRUE_OPT}')
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Policy Mean μ')
axes[1].set_title('Policy Mean Convergence', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('ch13_fig3_grpo_vs_reinforce.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Variance comparison over multiple seeds ───────────────────────────────
N_SEEDS = 10
final_mus = {m: [] for m in methods}

for seed in range(N_SEEDS):
    for method in methods:
        _, mus = run_bandit(method=method, seed=seed, n_steps=400)
        final_mus[method].append(mus[-1])

fig, ax = plt.subplots(figsize=(9, 4.5))

bp_data = [final_mus[m] for m in methods]
bp = ax.boxplot(bp_data, patch_artist=True, labels=labels,
                medianprops={'color': '#28251D', 'linewidth': 2})

for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)

ax.axhline(TRUE_OPT, color='#7A7974', linestyle='--', linewidth=1.5,
           label=f'True optimum μ={TRUE_OPT}')
ax.set_ylabel('Final Policy Mean μ (after 400 steps)')
ax.set_title(f'§13.4 — Variance of Convergence Across {N_SEEDS} Seeds', fontweight='bold')
ax.legend()
ax.grid(alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig('ch13_fig4_variance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('Final μ statistics:')
for method, label in zip(methods, labels):
    arr = np.array(final_mus[method])
    print(f'  {label:28s}: mean={arr.mean():.3f}, std={arr.std():.3f}')

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 20px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0 0 12px 0;">Сводка и ключевые выводы</h2>
  <ul style="color: #D4D1CA; line-height: 1.8; margin: 0; padding-left: 22px;">
    <li><strong>Преимущество GRPO</strong> нормализует каждое вознаграждение внутри своей группы, $A_i = (r_i - \bar{r}_g) / \text{std}(\mathbf{r}_g)$, давая преимущества с нулевым средним и единичным масштабом для более стабильных обновлений.</li>
    <li><strong>Размер группы $K$</strong> управляет стабильностью оценщика: большие группы уменьшают дисперсию преимуществ и сглаживают оценки градиента.</li>
    <li><strong>Вырожденность при $K=1$</strong> приводит к нулевому преимуществу, поэтому GRPO нужно как минимум два сэмпла на группу.</li>
    <li><strong>По сравнению с REINFORCE</strong>, GRPO показывает существенно меньшую дисперсию сходимости среднего стратегии на разных сидах в эксперименте с бандитом.</li>
    <li><strong>По сравнению с RLOO</strong>, GRPO жертвует несмещённой leave-one-out базовой линией ради нормализации на групповое стандартное отклонение, что удерживает масштаб преимуществ более согласованным.</li>
  </ul>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500; margin-top: 20px;">
  <strong style="color: #f0a500;">Дальше:</strong> в главе 14 всё связывается в практический RLHF-пайплайн на TRL.
</div>